# §13.6.4 — 같은 조건부 과제를 세 방식으로 풀기

> 딥러닝 교재 · 3부 13장 6절 4항 (🐍)
> 선행: §13.6.1(교차 어텐션) · §13.6.2(주입 방식의 사다리) · §13.6.5(선형 비용)

## 이 노트북이 답하는 질문

1. **요약 벡터·접두사·교차 어텐션은 같은 조건부 과제에서 어떻게 갈리는가?**
2. **갈림의 원인은 무엇인가?** 조건 길이 $S$를 훑어 병목을 드러낸다.
3. **비용 구조는 어떻게 다른가?** 파라미터와 스텝 시간을 실측한다.

**예상 실행 시간** CPU 약 4분 (`FAST = True`이면 약 2분).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 과제 — 조건의 일부를 골라 써야 하는 회상

조건은 키–값 쌍 $S$개, 질의는 키 하나, 정답은 그 키의 값이다(§12.6.5의 과제를 조건부
생성의 최소형으로 다시 쓴 것). 정답이 조건의 **어느 한 조각**에 있으므로, 조건을
벡터 하나로 접는 방식은 §12.6.1의 병목 논증을 정면으로 맞는다.

세 방식 모두 판독은 질의 조건부 양선형으로 통일하고, 다른 것은 **조건이 판독까지
오는 통로**뿐이다.

* **요약 벡터** — $\bar c=\frac1S\sum_j c_j$ 하나로 접어 전달.
* **접두사** — $[c_1,...,c_S,\,x_q]$를 한 열로 잇고 자기 어텐션 한 층. 질의 위치 출력이 판독으로.
* **교차 어텐션** — 질의가 조건 열을 직접 조회: $\alpha_j\propto\exp((W x_q)^\top c_j)$.

In [ ]:
N_K, N_V = 24, 8
D_IN = N_K + N_K * N_V

def make_batch(S, B, rn):
    Xc = np.zeros((B, S, D_IN))
    keys = np.array([rn.choice(N_K, size=S, replace=False) for _ in range(B)])
    vals = rn.integers(0, N_V, size=(B, S))
    Xc[np.arange(B)[:, None], np.arange(S)[None, :], keys] = 1.0
    Xc[np.arange(B)[:, None], np.arange(S)[None, :], N_K + keys * N_V + vals] = 1.0
    qpos = rn.integers(0, S, B)
    qkey = keys[np.arange(B), qpos]
    Q = np.zeros((B, N_K)); Q[np.arange(B), qkey] = 1.0
    y = vals[np.arange(B), qpos]
    return Xc, Q, y

D = 48

def init_model(mode, seed):
    rn = np.random.default_rng(seed)
    m = {'Wc': rn.standard_normal((D_IN, D)) / np.sqrt(D_IN) * 2.0,   # 조건 임베딩
         'Wqin': rn.standard_normal((N_K, D)) / np.sqrt(N_K),          # 질의 임베딩
         'A': rn.standard_normal((N_V, N_K, D)) / np.sqrt(D),          # 양선형 판독
         'bo': np.zeros(N_V)}
    if mode == 'prefix':
        for nm in ['Wq', 'Wk', 'Wv']:
            m[nm] = rn.standard_normal((D, D)) / np.sqrt(D)
    if mode == 'cross':
        m['Wx'] = rn.standard_normal((D, D)) / np.sqrt(D)
    return m

def forward_backward(m, mode, Xc, Q, y):
    B, S, _ = Xc.shape
    C = Xc @ m['Wc']                      # (B,S,D) 조건 임베딩
    xq = Q @ m['Wqin']                    # (B,D) 질의 임베딩
    aux = {}
    if mode == 'mean':
        feat = C.mean(1) + xq
    elif mode == 'cross':
        qv = xq @ m['Wx']
        e = np.einsum('bd,bsd->bs', qv, C); e -= e.max(1, keepdims=True)
        al = np.exp(e); al /= al.sum(1, keepdims=True)
        feat = np.einsum('bs,bsd->bd', al, C)
        aux = dict(qv=qv, al=al)
    else:                                  # prefix: 자기 어텐션 한 층
        Xall = np.concatenate([C, xq[:, None, :]], axis=1)          # (B,S+1,D)
        Qm, Km, Vm = Xall @ m['Wq'], Xall @ m['Wk'], Xall @ m['Wv']
        e = np.einsum('bd,bsd->bs', Qm[:, -1], Km) / np.sqrt(D)   # 질의 위치의 행만 필요
        e -= e.max(1, keepdims=True)
        al = np.exp(e); al /= al.sum(1, keepdims=True)
        feat = np.einsum('bs,bsd->bd', al, Vm) + xq                 # 잔차
        aux = dict(Xall=Xall, Qm=Qm, Km=Km, Vm=Vm, al=al)
    logits = np.einsum('bi,vij,bj->bv', Q, m['A'], feat) + m['bo']
    logits -= logits.max(1, keepdims=True)
    P = np.exp(logits); P /= P.sum(1, keepdims=True)
    loss = -np.mean(np.log(P[np.arange(B), y] + 1e-12))
    acc = np.mean(P.argmax(1) == y)
    # ── 역전파 ──
    g = {k: np.zeros_like(v) for k, v in m.items()}
    dlog = P.copy(); dlog[np.arange(B), y] -= 1; dlog /= B
    g['A'] = np.einsum('bv,bi,bj->vij', dlog, Q, feat); g['bo'] = dlog.sum(0)
    dfeat = np.einsum('bv,vij,bi->bj', dlog, m['A'], Q)
    dC = np.zeros_like(C); dxq = np.zeros_like(xq)
    if mode == 'mean':
        dC += dfeat[:, None, :] / S
        dxq += dfeat
    elif mode == 'cross':
        qv, al = aux['qv'], aux['al']
        dal = np.einsum('bd,bsd->bs', dfeat, C)
        dC += al[:, :, None] * dfeat[:, None, :]
        de = al * (dal - (al * dal).sum(1, keepdims=True))
        dqv = np.einsum('bs,bsd->bd', de, C)
        g['Wx'] = xq.T @ dqv
        dxq += dqv @ m['Wx'].T
        dC += de[:, :, None] * qv[:, None, :]
    else:
        Xall, Qm, Km, Vm, al = aux['Xall'], aux['Qm'], aux['Km'], aux['Vm'], aux['al']
        dxq += dfeat                                       # 잔차 가지
        dal = np.einsum('bd,bsd->bs', dfeat, Vm)
        dVm = al[:, :, None] * dfeat[:, None, :]
        de = al * (dal - (al * dal).sum(1, keepdims=True)) / np.sqrt(D)
        dQlast = np.einsum('bs,bsd->bd', de, Km)
        dKm = de[:, :, None] * Qm[:, -1][:, None, :]
        dXall = dVm @ m['Wv'].T + dKm @ m['Wk'].T
        dXall[:, -1] += dQlast @ m['Wq'].T
        g['Wq'] = np.einsum('bd,be->de', Xall[:, -1], dQlast)
        g['Wk'] = np.einsum('bsd,bse->de', Xall, dKm)
        g['Wv'] = np.einsum('bsd,bse->de', Xall, dVm)
        dC += dXall[:, :-1]; dxq += dXall[:, -1]
    g['Wc'] = np.einsum('bsd,bse->de', Xc, dC)
    g['Wqin'] = Q.T @ dxq
    return loss, acc, g, feat

def train(mode, S, seed=0, steps=None):
    steps = steps or (300 if FAST else 600)
    m = init_model(mode, seed)
    ms = {k: np.zeros_like(v) for k, v in m.items()}
    vs = {k: np.zeros_like(v) for k, v in m.items()}
    rb = np.random.default_rng(4000 + seed)
    t_step = []
    for t in range(1, steps + 1):
        Xc, Q, y = make_batch(S, 64, rb)
        tt = time.perf_counter()
        loss, acc, g, _ = forward_backward(m, mode, Xc, Q, y)
        t_step.append(time.perf_counter() - tt)
        for k in m:
            ms[k] = 0.9 * ms[k] + 0.1 * g[k]
            vs[k] = 0.999 * vs[k] + 0.001 * g[k] ** 2
            m[k] -= 3e-3 * (ms[k] / (1 - 0.9 ** t)) / (np.sqrt(vs[k] / (1 - 0.999 ** t)) + 1e-8)
    Xc, Q, y = make_batch(S, 1500, np.random.default_rng(SEED + 5))
    _, acc, _, _ = forward_backward(m, mode, Xc, Q, y)
    n_par = sum(v.size for v in m.values())
    return acc, n_par, np.median(t_step)

MODES = ['mean', 'prefix', 'cross']
MNAME = {'mean': lab('요약 벡터', 'mean'), 'prefix': lab('접두사', 'prefix'),
         'cross': lab('교차 어텐션', 'cross')}
print("모델 준비 완료")

---
## 2. 조건 길이 $S$를 훑는다

In [ ]:
S_LIST = [2, 8, 24] if FAST else [2, 4, 8, 16, 24]
res = {mo: {'acc': [], 'par': None, 'ts': []} for mo in MODES}
for mo in MODES:
    for S in S_LIST:
        acc, npar, ts = train(mo, S, seed=1)
        res[mo]['acc'].append(acc); res[mo]['par'] = npar; res[mo]['ts'].append(ts)
    print(f"{mo:7s}: 정확도 {np.round(res[mo]['acc'], 2)}  파라미터 {res[mo]['par']/1e3:.1f}k"
          f"  ({time.time()-_t0:.0f}초)")

---
## 3. 교재 그림 — fig_13_6_4

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()
MC = {'mean': CB[4], 'prefix': CB[1], 'cross': CB[5]}

# (a) 과제와 세 통로 — 도식
ax = axes[0]; ax.axis('off'); ax.set_xlim(0, 10); ax.set_ylim(0, 7)
for j in range(5):
    x0 = 0.5 + 1.15 * j
    ax.add_patch(plt.Rectangle((x0, 5.7), 1.0, 0.95, fc='#E8F4FB', ec='k', lw=0.8))
    ax.text(x0 + 0.5, 6.35, f'$k_{j+1}$', ha='center', va='center', fontsize=8)
    ax.text(x0 + 0.5, 5.93, f'$v_{j+1}$', ha='center', va='center', fontsize=8, color=CB[5])
ax.text(6.6, 6.15, '$\\cdots$', fontsize=11)
ax.add_patch(plt.Rectangle((7.3, 5.7), 1.7, 0.95, fc='#FDEBD9', ec=CB[4], lw=1.2))
ax.text(8.15, 6.17, lab('질의 $k_3$?', 'query'), ha='center', va='center', fontsize=9, color=CB[4])
ys = [3.9, 2.6, 1.3]
labels = [lab('요약 벡터:  $\\bar c=\\frac{1}{S}\\sum_j c_j$ — 접어서 전달', 'mean'),
          lab('접두사:  $[c_1,...,c_S,\\,x_q]$ 자기 어텐션 — 같은 통로에 합승', 'prefix'),
          lab('교차 어텐션:  $x_q$가 $c_{1..S}$를 직접 조회 — 전용 통로', 'cross')]
for yy, lb, mo in zip(ys, labels, MODES):
    ax.add_patch(plt.Rectangle((0.5, yy - 0.42), 8.9, 0.9, fc='white', ec=MC[mo], lw=1.2))
    ax.text(0.8, yy, lb, fontsize=9, va='center', color=MC[mo])
ax.annotate('', xy=(5.0, 4.55), xytext=(5.0, 5.62),
            arrowprops=dict(arrowstyle='->', color='0.5', lw=1.0))
ax.set_title(lab('(a) 같은 과제, 세 개의 통로', '(a) three routes'), fontsize=10)

# (b) 길이별 정확도
ax = axes[1]
for mo in MODES:
    ax.plot(S_LIST, res[mo]['acc'], 'o-', color=MC[mo], ms=5, label=MNAME[mo])
ax.axhline(1 / N_V, color='k', lw=0.7, ls=':')
ax.text(S_LIST[-1], 1 / N_V + 0.02, lab('우연 수준', 'chance'), fontsize=8, ha='right')
ax.set_ylim(0, 1.05)
ax.set_xlabel(lab('조건 길이 $S$', 'condition length'))
ax.set_ylabel(lab('회상 정확도', 'accuracy'))
ax.set_title(lab('(b) 접으면 무너지고, 열로 두면 버틴다', '(b) accuracy vs $S$'), fontsize=10)
ax.legend(fontsize=8)

# (c) 파라미터와 스텝 시간
ax = axes[2]
xs = np.arange(3); w = 0.38
pars = [res[mo]['par'] / 1e3 for mo in MODES]
ax.bar(xs - w / 2, pars, w, color=[MC[mo] for mo in MODES], alpha=0.9)
ax.set_xticks(xs); ax.set_xticklabels([MNAME[mo] for mo in MODES], fontsize=9)
ax.set_ylabel(lab('파라미터 (천 개)', 'params (k)'))
ax2 = ax.twinx()
ts24 = [res[mo]['ts'][-1] * 1e3 for mo in MODES]
ax2.plot(xs, ts24, 'D--', color='k', ms=6, lw=1)
ax2.set_ylabel(lab(f'스텝 시간 (ms, $S={S_LIST[-1]}$)', 'step time (ms)'))
ax2.grid(False)
ax.set_title(lab('(c) 비용 — 막대: 파라미터, 점선: 시간', '(c) cost'), fontsize=10)

# (d) 스텝 시간의 S 의존
ax = axes[3]
for mo in MODES:
    ax.plot(S_LIST, np.array(res[mo]['ts']) * 1e3, 'o-', color=MC[mo], ms=5, label=MNAME[mo])
ax.set_xlabel(lab('조건 길이 $S$', 'condition length'))
ax.set_ylabel(lab('스텝 시간 (ms)', 'step time (ms)'))
ax.set_title(lab('(d) 조건 길이에 비례하는 비용', '(d) cost vs $S$'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_13_6_4')
plt.show()

> ### 읽는 법
>
> (b) 요약 벡터는 조건이 짧을 때만 경쟁력이 있고 $S$가 자라면 병목으로 무너진다 —
> §12.6.5 곡선의 재연이다. 접두사와 교차 어텐션은 조건을 열로 보존하므로 버틴다.
> (c)–(d) 셋의 차이는 성능보다 비용의 구조에 있다. 요약 벡터는 $S$와 무관한 상수
> 비용, 교차 어텐션은 $O(S)$, 접두사는 자기 어텐션 길이가 $S{+}1$이 되므로 역시
> $S$에 비례해 자란다(깊은 모델이라면 $O((T{+}S)^2)$까지 간다, §13.6.5).
> "조건이 길면 통로 비용이 지배 항"이라는 §13.6.5의 경고가 이 두 패널의 요약이다.

---
## 5. 자기 점검

1. (b)에서 요약 벡터가 무너지기 시작하는 $S$를 §12.6.5(c)의 용량 사다리와 비교하라. 여기의 병목 폭은 무엇인가?
2. 접두사 방식에서 질의 위치의 어텐션 분포를 그려 보라. 정답 쌍 위치로 질량이 몰리는가? 교차 어텐션의 $\alpha$와 무엇이 다른가?
3. 조건이 생성 중 불변일 때 교차 어텐션의 $K,V$를 캐시하면 (d)의 곡선이 어떻게 변하는가? (§13.6.5의 처방)
4. 요약을 평균 대신 **학습된 어텐션 풀링**으로 바꾸면 (b)가 얼마나 회복되는가? 그 방식은 셋 중 어디에 가까워진 것인가?

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `S_LIST` | 2절 | ≤24 | 병목의 노출 정도 |
| `D` | 1절 | 48 | 요약 벡터의 병목 폭 |
| `N_V` | 1절 | 8 | 쌍당 정보량 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")